# Long Transaction: Session State Management

Deep-dive into the session state management that enables the iterative research pattern.
This pattern underpins the AGENTS.md harness, enabling quality-driven iteration.

## Learning Objectives
- Understand the ResearchSession data model
- Learn how state persists across iterations
- Practice saving and loading sessions from PostgreSQL

In [ ]:
import os
import sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from dotenv import load_dotenv
load_dotenv(os.path.join('..', '.env'))

from harness.session import ResearchSession, SessionManager
print("Harness session module loaded")

## 1. Creating a Research Session

In [ ]:
# Create a new session
session = ResearchSession(
    query="How does OpenShift AI integrate with Kagenti for multi-agent systems?",
    max_iterations=4,
    quality_threshold=7.5,
)

print(f"Session ID: {session.session_id}")
print(f"Query: {session.query}")
print(f"Status: {session.status}")
print(f"Should iterate: {session.should_iterate()}")

## 2. Simulating Iterations

In [ ]:
# Simulate iteration 1
session.status = "researching"
session.advance_iteration()
session.add_context("doc_search", "OpenShift AI provides model serving via vLLM...")
session.add_context("doc_search", "Kagenti is a Kubernetes-native agent control plane...")
session.current_draft = "Initial draft about RHOAI and Kagenti integration..."

# Verification result for iteration 1
session.quality_score = 4.5
session.add_verification({
    "score": 4.5,
    "passed": False,
    "feedback": "Too shallow, needs more technical detail",
})

print(f"After iteration 1:")
print(f"  Iteration: {session.iteration}")
print(f"  Quality: {session.quality_score}")
print(f"  Context items: {len(session.accumulated_context)}")
print(f"  Should iterate: {session.should_iterate()}")

In [ ]:
# Simulate iteration 2 (improved)
session.advance_iteration()
session.add_context("doc_search", "A2A protocol enables framework-agnostic communication...")
session.add_context("synthesis", "Integration flow: Kagenti manages identity and discovery...")
session.current_draft = "Improved draft with technical details and citations..."
session.quality_score = 7.8
session.add_verification({"score": 7.8, "passed": True})
session.mark_complete()

print(f"After iteration 2:")
print(f"  Iteration: {session.iteration}")
print(f"  Quality: {session.quality_score}")
print(f"  Context items: {len(session.accumulated_context)}")
print(f"  Status: {session.status}")
print(f"  Should iterate: {session.should_iterate()}")

## 3. Session Persistence

In [ ]:
# Save session to PostgreSQL
manager = SessionManager()
manager.ensure_table()
manager.save(session)
print(f"Session {session.session_id} saved to PostgreSQL")

# Load it back
loaded = manager.load(session.session_id)
print(f"\nLoaded session:")
print(f"  ID: {loaded.session_id}")
print(f"  Query: {loaded.query}")
print(f"  Status: {loaded.status}")
print(f"  Quality: {loaded.quality_score}")
print(f"  Iterations: {loaded.iteration}")

In [ ]:
# List recent sessions
sessions = manager.list_sessions(limit=5)
print("Recent sessions:")
for s in sessions:
    print(f"  [{s['session_id']}] {s['query'][:50]}... (status={s['status']}, score={s['quality_score']})")

## Summary

The long transaction pattern:
- **Versioned state** that evolves across iterations
- **PostgreSQL persistence** for resumability and auditing
- **Quality-driven termination** instead of fixed step counts
- **Context accumulation** that builds knowledge across passes

The long transaction is the state backbone of the AGENTS.md harness.